In [0]:
# openpyxl is required on Serverless compute for .xlsx parsing
%pip install openpyxl

import pandas as pd
from pyspark.sql.functions import current_date, current_timestamp, lit, regexp_extract, to_date

# Source: PSA landing zone Excel file
source_path = "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/facilities/primary_care_asof20260630.xlsx"

# Preserve every source row as received. Positional names keep title and header rows as data.
source_sheet = pd.ExcelFile(source_path).sheet_names[0]
source_file = source_path.rsplit('/', 1)[-1]
pdf = pd.read_excel(source_path, sheet_name=source_sheet, header=None)
pdf.columns = [f'_c{i}' for i in range(pdf.shape[1])]

df = (
    spark.createDataFrame(pdf)
         .withColumn('source_file', lit(source_file))
         .withColumn('as_of_date', to_date(regexp_extract(lit(source_file), r'(?i)asof_?(\d{8})', 1), 'yyyyMMdd'))
         .withColumn('extract_date', current_date())
         .withColumn('ingested_at', current_timestamp())
)

# Write the raw source with technical ingestion metadata.
df.write.format("delta") \
  .mode("overwrite") \
  .option("delta.columnMapping.mode", "name") \
  .option("overwriteSchema", "true") \
  .saveAsTable("public_health.bronze.facilities_primary_care")
